In [1]:
import json
import os
import time
import requests
import pandas as pd
from typing import Any
from dotenv import load_dotenv

load_dotenv()

def gemini_api(guideline: str, data: Any) -> str:
    try:
        API_KEY = os.environ.get("GEMINI_API_KEY")
        if not API_KEY:
            raise ValueError("❌ GEMINI_API_KEY가 없어요. .env 파일을 확인해주세요.")

        url = "https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent"
        headers = {
            "x-goog-api-key": API_KEY,
            "Content-Type": "application/json",
        }

        if isinstance(data, str):
            data_text = data
        else:
            try:
                data_text = json.dumps(data, ensure_ascii=False, default=str)
            except Exception:
                data_text = str(data)

        payload = {
            "systemInstruction": {
                "role": "system",
                "parts": [{"text": guideline}],
            },
            "contents": [
                {
                    "role": "user",
                    "parts": [{"text": data_text}],
                }
            ],
        }

        resp = requests.post(url, headers=headers, json=payload, timeout=60)
        resp.raise_for_status()
        out = resp.json()

        candidates = out.get("candidates") or []
        if not candidates:
            raise ValueError(f"No candidates in response: {out}")

        content = (candidates[0].get("content") or {})
        parts = content.get("parts") or []
        text = "".join((p.get("text") or "") for p in parts).strip()

        if not text:
            raise ValueError(f"Empty text in response: {out}")

        return text
    except Exception as e:
        print(f"  [API 오류] {e}")
        return None


지침 = '''
# 리뷰 감성분석

## 역할
너는 한국어 리뷰 감성분석 전문가야.

## 입력 형식
여러 개의 리뷰가 아래 JSON 배열로 제공돼:
[{"id": 0, "review": "리뷰 내용"}, ...]

## 분석 기준
- score: 1~5점 (1=매우부정, 2=부정, 3=중립, 4=긍정, 5=매우긍정)
- 5점은 완전한 긍정일 때만 부여 (보수적으로 판단)
- sentiment: score 기준으로 positive / negative / neutral 분류
  - 4~5점 → positive
  - 3점   → neutral
  - 1~2점 → negative

## 출력 형식
- 반드시 JSON 배열로만 응답
- 입력된 id를 그대로 유지
- JSON 외 추가 설명 절대 금지
- 마크다운 코드블록(```) 사용 금지

[
  {"id": 0, "score": 4, "sentiment": "positive", "reason": "판단 근거 한 줄"},
  {"id": 1, "score": 2, "sentiment": "negative", "reason": "판단 근거 한 줄"}
]

## 예시
입력: [{"id": 0, "review": "배송은 빠른데 품질이 너무 별로예요"}]
출력: [{"id": 0, "score": 2, "sentiment": "negative", "reason": "품질 불만이 핵심으로 부정적 평가"}]
'''


def analyze_batch(reviews: list[dict]) -> list[dict]:
    result_text = gemini_api(지침, reviews)
    if result_text is None:
        return []
    try:
        clean = result_text.replace("```json", "").replace("```", "").strip()
        parsed = json.loads(clean)

        if isinstance(parsed, dict):
            for key in ["results", "data", "items"]:
                if key in parsed:
                    parsed = parsed[key]
                    break

        if isinstance(parsed, dict):
            parsed = [parsed]

        for i, item in enumerate(parsed):
            if "id" not in item:
                item["id"] = reviews[i]["id"]

        return parsed

    except json.JSONDecodeError as e:
        print(f"  [파싱 오류] {e}")
        print(f"  응답 내용: {result_text[:200]}")
        return []


def save_checkpoint(results: dict, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2, default=str)

def load_checkpoint(path: str) -> dict:
    if os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8") as f:
                raw = json.load(f)
            return {int(k): v for k, v in raw.items() if k.isdigit()}
        except json.JSONDecodeError:
            print("⚠️ checkpoint.json이 손상되었어요. 백업 후 초기화합니다.")
            import shutil
            shutil.copy(path, path + ".broken")
            print(f"   백업 저장: {path}.broken")
            return {}
    return {}

print("✅ 셀 1 완료 - 함수 로드됨")

✅ 셀 1 완료 - 함수 로드됨


In [3]:
INPUT_PATH      = "naver_lg.csv"
OUTPUT_PATH     = "naver_lg_감성결과.csv"
REVIEW_COL      = "clean_review"
BATCH_SIZE      = 20
RUN_SIZE        = 1000
DELAY           = 1.5
MAX_RETRY       = 3
CHECKPOINT_PATH = "checkpoint.json"

df = pd.read_csv(INPUT_PATH)
if REVIEW_COL not in df.columns:
    raise ValueError(f"❌ '{REVIEW_COL}' 열이 없어요.")

results = load_checkpoint(CHECKPOINT_PATH)
start_idx = len(results)
end_idx   = min(start_idx + RUN_SIZE, len(df))

if start_idx >= len(df):
    print("🎉 이미 전체 완료!")
else:
    print(f"🚀 실행 구간: {start_idx} ~ {end_idx-1} ({end_idx - start_idx}건)")

    target_df = df.iloc[start_idx:end_idx]
    batches   = [target_df.iloc[i:i+BATCH_SIZE] for i in range(0, len(target_df), BATCH_SIZE)]

    for batch_idx, batch_df in enumerate(batches):
        print(f"  [{batch_idx+1}/{len(batches)}배치] idx {batch_df.index[0]}~{batch_df.index[-1]}")

        batch_input = [
            {"id": idx, "review": row[REVIEW_COL]}
            for idx, row in batch_df.iterrows()
        ]

        batch_results = []
        for attempt in range(MAX_RETRY):
            batch_results = analyze_batch(batch_input)
            if batch_results:
                break
            print(f"  ⚠️ 재시도 {attempt+1}/{MAX_RETRY}...")
            time.sleep(3)

        if batch_results:
            for item in batch_results:
                idx = item["id"]
                row = df.loc[idx]
                results[idx] = {
                    "score":        item.get("score"),
                    "sentiment":    item.get("sentiment"),
                    "reason":       item.get("reason"),
                    "product_id":   row.get("product_id"),
                    "product_name": row.get("product_name"),
                    "price":        row.get("price"),
                    "clean_review": row.get("clean_review"),
                }
            print(f"  ✅ 완료 | 누적: {len(results)}/{len(df)}건")
        else:
            print(f"  ❌ 실패 - None 저장")
            for idx in batch_df.index:
                row = df.loc[idx]
                results[idx] = {
                    "score":        None,
                    "sentiment":    None,
                    "reason":       None,
                    "product_id":   row.get("product_id"),
                    "product_name": row.get("product_name"),
                    "price":        row.get("price"),
                    "clean_review": row.get("clean_review"),
                }

        time.sleep(DELAY)

    save_checkpoint(results, CHECKPOINT_PATH)
    print(f"\n💾 체크포인트 저장 ({len(results)}/{len(df)}건)")
    print(f"📌 남은 건수: {len(df) - len(results)}건 → 다음 실행 시 이어서 진행")

🚀 실행 구간: 1000 ~ 1999 (1000건)
  [1/50배치] idx 1000~1019
  ✅ 완료 | 누적: 1020/4924건
  [2/50배치] idx 1020~1039
  ✅ 완료 | 누적: 1040/4924건
  [3/50배치] idx 1040~1059
  ✅ 완료 | 누적: 1060/4924건
  [4/50배치] idx 1060~1079
  [파싱 오류] Expecting ':' delimiter: line 10 column 16 (char 823)
  응답 내용: [
  {"id": 1060, "score": 4, "sentiment": "positive", "reason": "선물을 받은 사람이 만족하고 추천함."},
  {"id": 1061, "score": 5, "sentiment": "positive", "reason": "제품이 최적이며 배송, 구매 조건, 고객 응대 모두 만점이라고 언급함."},
  {"i
  ⚠️ 재시도 1/3...
  ✅ 완료 | 누적: 1080/4924건
  [5/50배치] idx 1080~1099
  ✅ 완료 | 누적: 1100/4924건
  [6/50배치] idx 1100~1119
  ✅ 완료 | 누적: 1120/4924건
  [7/50배치] idx 1120~1139
  ✅ 완료 | 누적: 1140/4924건
  [8/50배치] idx 1140~1159
  ✅ 완료 | 누적: 1160/4924건
  [9/50배치] idx 1160~1179
  ✅ 완료 | 누적: 1180/4924건
  [10/50배치] idx 1180~1199
  ✅ 완료 | 누적: 1200/4924건
  [11/50배치] idx 1200~1219
  [파싱 오류] Expecting ':' delimiter: line 16 column 16 (char 1509)
  응답 내용: [
  {"id": 1200, "score": 5, "sentiment": "positive", "reason": "전반적으로 매우 편리하고 빠르며 구성품도 잘 